Paquetes necesarios

In [ ]:
import cv2  
import csv
from ultralytics import YOLO
from paddleocr import PaddleOCR
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
import re

c:\Users\skyri\anaconda3\envs\VC_P4\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Carga del modelo, descarga en disco si no está presente en la carpeta
model = YOLO('yolo11n.pt') # Contenedores

# Etiqueta de las distintas clases
classNames = ["person", "bicycle", "car"]

#Para un vídeo 
filename = "assets/C0142.mp4"
results = model(source=filename,
                show=True,
                imgsz=320)

cv2.destroyAllWindows()


### Codigo basico sin OCR

In [7]:
# Carga del modelo, descarga en disco si no está presente en la carpeta
model = YOLO('yolo11n.pt') # Contenedores
model2 = YOLO('matriculas_best.pt') # Matrículas

classNames = ["person", "bicycle", "car", "motorcycle", "truck", "bus"]
clases_vehiculos = [2, 3, 5, 7]
clases_yolo = [0, 2, 3, 5, 7]

cap = cv2.VideoCapture("assets/C0142.mp4")

total_clases = {
    0: set(),  # person
    2: set(),  # coches
    3: set(),  # motos
    5: set(),  # trucks
    7: set()   # buses
}

ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

output_video = cv2.VideoWriter('output_video.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (ancho, alto))

frame_count = 0

csv_file = open('detections.csv', mode='w', newline='')
writer = csv.writer(csv_file)
writer.writerow(['frame', 'class', 'confidence', 'track_id', 'x1', 'y1', 'x2', 'y2', 'matrícula_en_su_caso', 'confianza', 'mx1', 'my1', 'mx2', 'my2', 'texto_matricula'])


while True:
    ret, frame = cap.read()
    frame_count += 1
    
    if not ret:
        break

    # Para cada detección
    results = model.track(source=frame, persist=True, imgsz=320, verbose=False, classes=clases_yolo)
    
    for box in results[0].boxes:
        # Contenedor
        x1, y1, x2, y2 = box.xyxy[0]
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2) 

        # Clase
        cls = int(box.cls[0])

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2) # color verde para el contenedor
        cv2.putText(frame, model.names[cls] , (x1, max(15, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    
        # Parametros Csv
        conf_model = float(box.conf[0])
        objeto = model.names[cls]

        track_id = -1  # Valor por defecto si no hay ID de seguimiento
        if box.id is not None:
            track_id = int(box.id[0])

            if cls in total_clases:
                total_clases[cls].add(track_id)


        tiene_mat = 0
        conf_mat = 0.0
        g_x1, g_y1, g_x2, g_y2 = 0, 0, 0, 0
        texto_matricula = ""

        if cls in clases_vehiculos:
            vehiculo_recortado = frame[max(0, y1):max(0, y2), max(0, x1):max(0, x2)]
            
            results2 = model2(source=vehiculo_recortado, imgsz=320)

            if len(results2[0].boxes) > 0:
                for box2 in results2[0].boxes:
                    print(f"El modelo 2 ha detectado: {model2.names[int(box2.cls[0])]}")

                    if int(box2.cls[0]) == 0:  # Si la clase detectada es matrícula
                        tiene_mat = 1
                        conf_mat = float(box2.conf[0])

                        x1_2, y1_2, x2_2, y2_2 = box2.xyxy[0]
                        x1_2, y1_2, x2_2, y2_2 = int(x1_2), int(y1_2), int(x2_2), int(y2_2)

                        g_x1, g_y1 = x1 + x1_2, y1 + y1_2
                        g_x2, g_y2 = x1 + x2_2, y1 + y2_2

                        cv2.rectangle(frame, (g_x1, g_y1), (g_x2, g_y2), (255, 0, 0), 2) # color azul para la matrícula

        writer.writerow([
            frame_count, objeto, round(conf_model, 2), track_id,
            x1, y1, x2, y2,
            tiene_mat, round(conf_mat, 2), g_x1, g_y1, g_x2, g_y2, texto_matricula
        ])
    output_video.write(frame)

    cv2.imshow("Detección de vehículos", frame)
    if cv2.waitKey(20) == 27:
        break

for cls, boxes in total_clases.items():
    print(f"Clase {cls}: {len(boxes)} detecciones")

cv2.destroyAllWindows()
output_video.release()
cap.release()
csv_file.close()


0: 288x320 1 plate, 8.7ms
Speed: 0.6ms preprocess, 8.7ms inference, 1.4ms postprocess per image at shape (1, 3, 288, 320)
El modelo 2 ha detectado: plate

0: 288x320 1 plate, 9.0ms
Speed: 0.6ms preprocess, 9.0ms inference, 1.4ms postprocess per image at shape (1, 3, 288, 320)
El modelo 2 ha detectado: plate

0: 288x320 1 plate, 8.7ms
Speed: 0.6ms preprocess, 8.7ms inference, 1.2ms postprocess per image at shape (1, 3, 288, 320)
El modelo 2 ha detectado: plate

0: 288x320 1 plate, 8.6ms
Speed: 0.9ms preprocess, 8.6ms inference, 1.2ms postprocess per image at shape (1, 3, 288, 320)
El modelo 2 ha detectado: plate

0: 288x320 1 plate, 8.7ms
Speed: 0.6ms preprocess, 8.7ms inference, 1.2ms postprocess per image at shape (1, 3, 288, 320)
El modelo 2 ha detectado: plate

0: 256x320 1 plate, 9.5ms
Speed: 0.7ms preprocess, 9.5ms inference, 1.2ms postprocess per image at shape (1, 3, 256, 320)
El modelo 2 ha detectado: plate

0: 256x320 1 plate, 9.0ms
Speed: 0.6ms preprocess, 9.0ms inference, 1

#### Codigo con PaddleOCR

In [ ]:



model = YOLO('yolo11n.pt')
model2 = YOLO('matriculas_best.pt')
ocr = PaddleOCR(
    lang='es',
    enable_mkldnn=False,
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False
)

clases_vehiculos = [2, 3, 5, 7]
clases_yolo = [0, 2, 3, 5, 7]
patrones_matricula = [re.compile(r'\d{4}[A-Z]{3}'), re.compile(r'[A-Z]{1,2}\d{4}[A-Z]{1,2}')]
max_intentos = 20
ancho_minimo = 30
ultimo_ancho = {}
crudos = []


def extraer_matricula(texto):
    for patron in patrones_matricula:
        coincidencia = patron.search(texto)
        if coincidencia:
            return coincidencia.group(0)
    return ""


def preparar_recorte(img):
    factor = max(1.0, 80 / img.shape[0])
    img = cv2.resize(img, None, fx=factor, fy=factor, interpolation=cv2.INTER_CUBIC)
    return cv2.copyMakeBorder(img, 10, 10, 10, 10, cv2.BORDER_REPLICATE)


def dibujar_etiqueta(img, texto, x, y):
    (w, h), base = cv2.getTextSize(texto, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)
    x = max(0, min(x, img.shape[1] - w - 8))
    arriba = max(0, y - h - base - 8)
    cv2.rectangle(img, (x, arriba), (x + w + 8, arriba + h + base + 8), (255, 0, 0), -1)
    cv2.putText(img, texto, (x + 4, arriba + h + 4), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)


cap = cv2.VideoCapture("assets/C0142.mp4")

total_clases = {
    0: set(),
    2: set(),
    3: set(),
    5: set(),
    7: set()
}

ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

output_video = cv2.VideoWriter('output_video.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (ancho, alto))

frame_count = 0
lecturas = {}
intentos = {}

csv_file = open('detections.csv', mode='w', newline='')
writer = csv.writer(csv_file)
writer.writerow(['frame', 'class', 'confidence', 'track_id', 'x1', 'y1', 'x2', 'y2', 'matrícula_en_su_caso', 'confianza', 'mx1', 'my1', 'mx2', 'my2', 'texto_matricula'])

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    limpio = frame.copy()

    results = model.track(source=frame, persist=True, imgsz=320, verbose=False,classes=clases_yolo)

    for box in results[0].boxes:
        x1, y1, x2, y2 = (int(v) for v in box.xyxy[0])
        x1, y1 = max(0, min(x1, ancho - 1)), max(0, min(y1, alto - 1))
        x2, y2 = max(0, min(x2, ancho)), max(0, min(y2, alto))

        cls = int(box.cls[0])

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, model.names[cls], (x1, max(15, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        conf_model = float(box.conf[0])
        objeto = model.names[cls]

        track_id = -1
        if box.id is not None:
            track_id = int(box.id[0])

            if cls in total_clases:
                total_clases[cls].add(track_id)

        tiene_mat = 0
        conf_mat = 0.0
        g_x1, g_y1, g_x2, g_y2 = 0, 0, 0, 0
        texto_matricula = ""

        if cls in clases_vehiculos and x2 > x1 and y2 > y1:
            results2 = model2(source=limpio[y1:y2, x1:x2], imgsz=640, conf=0.15, verbose=False)

            if len(results2[0].boxes) > 0:
                box2 = results2[0].boxes[int(results2[0].boxes.conf.argmax())]

                tiene_mat = 1
                conf_mat = float(box2.conf[0])

                x1_2, y1_2, x2_2, y2_2 = (int(v) for v in box2.xyxy[0])

                g_x1, g_y1 = x1 + x1_2, y1 + y1_2
                g_x2, g_y2 = x1 + x2_2, y1 + y2_2

                if track_id != -1:
                    texto_matricula = lecturas.get(track_id, "")

                ancho_placa = g_x2 - g_x1

                if (texto_matricula == "" and ancho_placa >= ancho_minimo
                        and ancho_placa >= 1.2 * ultimo_ancho.get(track_id, 0)
                        and (track_id == -1 or intentos.get(track_id, 0) < max_intentos)):
                    ultimo_ancho[track_id] = ancho_placa
                    matricula_recortada = limpio[g_y1:g_y2, g_x1:g_x2]

                    if matricula_recortada.size > 0:
                        matricula_recortada = preparar_recorte(matricula_recortada)
                        result = ocr.predict(matricula_recortada)[0]
                        crudo = re.sub(r'[^A-Z0-9]', '', ''.join(result['rec_texts']).upper())
                        crudos.append(crudo)
                        texto_matricula = extraer_matricula(crudo)

                        if track_id != -1:
                            if texto_matricula:
                                lecturas[track_id] = texto_matricula
                            else:
                                intentos[track_id] = intentos.get(track_id, 0) + 1

                cv2.rectangle(frame, (g_x1, g_y1), (g_x2, g_y2), (255, 0, 0), 2)

                if texto_matricula != "":
                    dibujar_etiqueta(frame, texto_matricula, g_x1, g_y1)

        writer.writerow([
            frame_count, objeto, round(conf_model, 2), track_id,
            x1, y1, x2, y2,
            tiene_mat, round(conf_mat, 2), g_x1, g_y1, g_x2, g_y2, texto_matricula
        ])
    output_video.write(frame)

    cv2.imshow("Detección de vehículos", frame)
    if cv2.waitKey(1) == 27:
        break

for cls, boxes in total_clases.items():
    print(f"Clase {cls} ({model.names[cls]}): {len(boxes)} detecciones")
print("Matrículas leídas:", len(lecturas))
print(crudos[:30])

cv2.destroyAllWindows()
output_video.release()
cap.release()
csv_file.close()

c:\Users\skyri\anaconda3\envs\VC_P4\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv6_medium_det', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-OCRv6_medium_det), the model files will be automatically downloaded and saved in `C:\Users\skyri\.paddlex\official_models\PP-OCRv6_medium_det`.
Fetching 5 files: 100%|██████████| 5/5 [00:07<00:00,  1.52s/it]
Creating model: ('PP-OCRv6_medium_rec', None, None)
Using official model (PP-OCRv6_medium_rec), the model files will be automatically downloaded and saved in `C:\Users\skyri\.paddlex\official_models\PP-OCRv6_medium_rec`.
Fetching 5 fi

Clase 0 (person): 4 detecciones
Clase 2 (car): 7 detecciones
Clase 3 (motorcycle): 0 detecciones
Clase 5 (bus): 1 detecciones
Clase 7 (truck): 1 detecciones
Matrículas leídas: 2
['332', '3324', '3324H2C', '', '5724', 'ST44GEP', '', '88713183', '3713LKC', '', '', '33K2', '', '', 'ES', 'N', '2008XCS']


#### Codigo con VLMs

In [8]:

model = YOLO('yolo11n.pt')
model2 = YOLO('matriculas_best.pt')
Device = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if Device == "cuda" else torch.float32
VLM_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"

processor = AutoProcessor.from_pretrained(VLM_ID)
model_ocr = AutoModelForImageTextToText.from_pretrained(
    VLM_ID, dtype=DTYPE, _attn_implementation="sdpa"
).to(Device)


classNames = ["person", "bicycle", "car", "motorcycle", "truck", "bus"]
clases_vehiculos = [2, 3, 5, 7]
clases_yolo = [0, 2, 3, 5, 7]
patrones_matricula = [re.compile(r'\d{4}[A-Z]{3}'), re.compile(r'[A-Z]{1,2}\d{4}[A-Z]{1,2}')]
max_intentos = 20
ancho_minimo = 30
ultimo_ancho = {}
crudos = []

PROMPT = "Read the license plate in this image. Answer only with the letters and numbers of the plate."
 

def leer_matricula_vlm(img):
    imagen = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    mensajes = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT}]}]
    prompt = processor.apply_chat_template(mensajes, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[imagen], return_tensors="pt").to(Device, dtype=DTYPE)
 
    with torch.no_grad():
        salida = model_ocr.generate(**inputs, max_new_tokens=15, do_sample=False)
    
    nuevos = salida[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(nuevos, skip_special_tokens=True)[0].strip()


def extraer_matricula(texto):
    for patron in patrones_matricula:
        coincidencia = patron.search(texto)
        if coincidencia:
            return coincidencia.group(0)
    return ""


def preparar_recorte(img):
    factor = max(1.0, 80 / img.shape[0])
    img = cv2.resize(img, None, fx=factor, fy=factor, interpolation=cv2.INTER_CUBIC)
    return cv2.copyMakeBorder(img, 10, 10, 10, 10, cv2.BORDER_REPLICATE)


def dibujar_etiqueta(img, texto, x, y):
    (w, h), base = cv2.getTextSize(texto, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)
    x = max(0, min(x, img.shape[1] - w - 8))
    arriba = max(0, y - h - base - 8)
    cv2.rectangle(img, (x, arriba), (x + w + 8, arriba + h + base + 8), (255, 0, 0), -1)
    cv2.putText(img, texto, (x + 4, arriba + h + 4), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)


cap = cv2.VideoCapture("assets/C0142.mp4")

total_clases = {
    0: set(),
    2: set(),
    3: set(),
    5: set(),
    7: set()
}

ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

output_video = cv2.VideoWriter('output_video.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (ancho, alto))

frame_count = 0
lecturas = {}
intentos = {}

csv_file = open('detections.csv', mode='w', newline='')
writer = csv.writer(csv_file)
writer.writerow(['frame', 'class', 'confidence', 'track_id', 'x1', 'y1', 'x2', 'y2', 'matrícula_en_su_caso', 'confianza', 'mx1', 'my1', 'mx2', 'my2', 'texto_matricula'])

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    limpio = frame.copy()

    results = model.track(source=frame, persist=True, imgsz=320, verbose=False,classes=clases_yolo)

    for box in results[0].boxes:
        x1, y1, x2, y2 = (int(v) for v in box.xyxy[0])
        x1, y1 = max(0, min(x1, ancho - 1)), max(0, min(y1, alto - 1))
        x2, y2 = max(0, min(x2, ancho)), max(0, min(y2, alto))

        cls = int(box.cls[0])

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, model.names[cls], (x1, max(15, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        conf_model = float(box.conf[0])
        objeto = model.names[cls]

        track_id = -1
        if box.id is not None:
            track_id = int(box.id[0])

            if cls in total_clases:
                total_clases[cls].add(track_id)

        tiene_mat = 0
        conf_mat = 0.0
        g_x1, g_y1, g_x2, g_y2 = 0, 0, 0, 0
        texto_matricula = ""

        if cls in clases_vehiculos and x2 > x1 and y2 > y1:
            results2 = model2(source=limpio[y1:y2, x1:x2], imgsz=640, conf=0.15, verbose=False)

            if len(results2[0].boxes) > 0:
                box2 = results2[0].boxes[int(results2[0].boxes.conf.argmax())]

                tiene_mat = 1
                conf_mat = float(box2.conf[0])

                x1_2, y1_2, x2_2, y2_2 = (int(v) for v in box2.xyxy[0])

                g_x1, g_y1 = x1 + x1_2, y1 + y1_2
                g_x2, g_y2 = x1 + x2_2, y1 + y2_2

                if track_id != -1:
                    texto_matricula = lecturas.get(track_id, "")

                ancho_placa = g_x2 - g_x1

                if (texto_matricula == "" and ancho_placa >= ancho_minimo
                        and ancho_placa >= 1.2 * ultimo_ancho.get(track_id, 0)
                        and (track_id == -1 or intentos.get(track_id, 0) < max_intentos)):
                    ultimo_ancho[track_id] = ancho_placa
                    matricula_recortada = limpio[g_y1:g_y2, g_x1:g_x2]

                    if matricula_recortada.size > 0:
                        matricula_recortada = preparar_recorte(matricula_recortada)
                        result = leer_matricula_vlm(matricula_recortada)
                        crudo = re.sub(r'[^A-Z0-9]', '', result.upper())
                        crudos.append(crudo)
                        texto_matricula = extraer_matricula(crudo)

                        if track_id != -1:
                            if texto_matricula:
                                lecturas[track_id] = texto_matricula
                            else:
                                intentos[track_id] = intentos.get(track_id, 0) + 1

                cv2.rectangle(frame, (g_x1, g_y1), (g_x2, g_y2), (255, 0, 0), 2)

                if texto_matricula != "":
                    dibujar_etiqueta(frame, texto_matricula, g_x1, g_y1)

        writer.writerow([
            frame_count, objeto, round(conf_model, 2), track_id,
            x1, y1, x2, y2,
            tiene_mat, round(conf_mat, 2), g_x1, g_y1, g_x2, g_y2, texto_matricula
        ])
    output_video.write(frame)

    cv2.imshow("Detección de vehículos", frame)
    if cv2.waitKey(1) == 27:
        break

for cls, boxes in total_clases.items():
    print(f"Clase {cls} ({model.names[cls]}): {len(boxes)} detecciones")
print("Matrículas leídas:", len(lecturas))
print(crudos[:30])

cv2.destroyAllWindows()
output_video.release()
cap.release()
csv_file.close()

Loading weights: 100%|██████████| 471/471 [00:00<00:00, 2767.87it/s]


Clase 0 (person): 4 detecciones
Clase 2 (car): 5 detecciones
Clase 3 (motorcycle): 0 detecciones
Clase 5 (bus): 1 detecciones
Clase 7 (truck): 1 detecciones
Matrículas leídas: 1
['2222', '0122345', '3225', '00000000', '000000000', '579000', '000000000', '1234567890', '713000', '3713LKC', '12345', '1234567890', 'N1995', '1234567890', '1234567890']
